<a href="https://colab.research.google.com/github/05050505050505/diary-django/blob/master/livedoor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **少数データにおけるバイアスを抑えるやつ**


# **データ準備**

まずは、データの準備をする。csvファイル

Jupyter/Colab における「強制的に /content フォルダの中身を根こそぎ削除する」シェルコマンド
一番最初のコードは全部消して一からやり直すためのものなので使用厳禁！！

In [1]:
!rm -rf /content/*

wget というコマンド‐ライン用ダウンローダーを呼び出して指定した URL（ここでは rondhuit が公開している livedoor ニュースコーパスのアーカイブ）をカレントディレクトリにそのままダウンロードする

In [3]:
!wget https://www.rondhuit.com/download/ldcc-20140209.tar.gz

--2025-06-12 04:43:27--  https://www.rondhuit.com/download/ldcc-20140209.tar.gz
Resolving www.rondhuit.com (www.rondhuit.com)... 59.106.19.174
Connecting to www.rondhuit.com (www.rondhuit.com)|59.106.19.174|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8855190 (8.4M) [application/x-gzip]
Saving to: ‘ldcc-20140209.tar.gz’

ldcc-20140209.tar.g 100%[===================>]   8.44M  1.95MB/s    in 7.1s    

2025-06-12 04:43:35 (1.19 MB/s) - ‘ldcc-20140209.tar.gz’ saved [8855190/8855190]



In [4]:
import os
import pandas as pd
import tarfile

# ファイルパスを指定する
tar_file_path = "/content/ldcc-20140209.tar.gz"
extract_folder = "/content/ldcc_data/"

# tar.gzファイルを解凍し、extract_folderに格納する
with tarfile.open(tar_file_path, "r:gz") as tar:
    tar.extractall(path=extract_folder)#（extract_folder）に圧縮ファイルの内容を展開して保存する処理

In [5]:
import glob

# 解凍されたフォルダ内のすべてのテキストファイルを取得
file_paths = glob.glob(extract_folder + "text/*/*.txt")
articles = []

# 各ファイルを開いてカテゴリ、タイトル、本文を抽出
for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        category = file_path.split("/")[-2]  # カテゴリ名を取得
        title = lines[2].strip()             # タイトルが3行目にある
        body = "".join(lines[3:]).strip()    # 本文が4行目以降

        articles.append([category, title, body])

# データフレームに変換
df = pd.DataFrame(articles, columns=["category", "title", "body"])

# データの先頭5行を表示して確認
df.head()


,category,title,body
0,dokujo-tsushin,8割の女性が悩む「お腹の張り」 スッキリ“腸内美人”を目指すには？,気付くとお腹がポッコリ、シルエットが気になってタイトな洋服が着られない…。独女なら1度は感じ...
1,dokujo-tsushin,離婚歴のある男性との交際、過去はやっぱり気になるもの？,現在独身でも、過去に「離婚歴がある」という人は、年代限らず珍しくない時代である。ちなみに厚生...
2,dokujo-tsushin,女の本音全開！「彼氏にドン引きした○○な瞬間」,それ相応に恋愛経験を積んできたであろう大人の皆さま方、あなたは過去につきあっていた恋人の言動...
3,dokujo-tsushin,【独女通信編集部】2011年記事まとめ vol.2,2011年もいよいよ大詰め。\n\n今年1月から配信した独女通信の全332本の記事の中から、...
4,dokujo-tsushin,くさったよめがあらわれた！vol.03「いつも私と同じことばかり考えてる君が好き！」 pre...,独女の皆さんこんにちは！\n毎度おなじみ（？）「くさったよめがあらわれた」の漫画家・うえやま...


In [6]:
# 全体のデータ数
total_count = len(df)

# カテゴリごとの記事数
category_counts = df['category'].value_counts()

# タイトルと本文の平均文字数
title_length_avg = df['title'].apply(len).mean()
body_length_avg = df['body'].apply(len).mean()

# 結果の表示
print("データセットの概要:")
print(f"全体のデータ数: {total_count} 件")
print("\nカテゴリごとの記事数:")
print(category_counts)
print(f"\nタイトルの平均文字数: {title_length_avg:.2f} 字")
print(f"本文の平均文字数: {body_length_avg:.2f} 字")


データセットの概要:
全体のデータ数: 7376 件

カテゴリごとの記事数:
category
sports-watch      901
dokujo-tsushin    871
it-life-hack      871
smax              871
movie-enter       871
kaden-channel     865
peachy            843
topic-news        771
livedoor-homme    512
Name: count, dtype: int64

タイトルの平均文字数: 37.68 字
本文の平均文字数: 1220.01 字


In [7]:
# 前処理 - 記号の除去、半角・全角の統一
import re

# タイトルと本文の記号を除去、全角を半角に変換
df['title'] = df['title'].apply(lambda x: re.sub(r'[^\w\s]', '', x).replace("　", " "))
df['body'] = df['body'].apply(lambda x: re.sub(r'[^\w\s]', '', x).replace("　", " "))

# 確認
df.head()

,category,title,body
0,dokujo-tsushin,8割の女性が悩むお腹の張り スッキリ腸内美人を目指すには,気付くとお腹がポッコリシルエットが気になってタイトな洋服が着られない独女なら1度は感じたこと...
1,dokujo-tsushin,離婚歴のある男性との交際過去はやっぱり気になるもの,現在独身でも過去に離婚歴があるという人は年代限らず珍しくない時代であるちなみに厚生労働省発表...
2,dokujo-tsushin,女の本音全開彼氏にドン引きしたな瞬間,それ相応に恋愛経験を積んできたであろう大人の皆さま方あなたは過去につきあっていた恋人の言動に...
3,dokujo-tsushin,独女通信編集部2011年記事まとめ vol2,2011年もいよいよ大詰め\n\n今年1月から配信した独女通信の全332本の記事の中から先に...
4,dokujo-tsushin,くさったよめがあらわれたvol03いつも私と同じことばかり考えてる君が好き presente...,独女の皆さんこんにちは\n毎度おなじみくさったよめがあらわれたの漫画家うえやま洋介犬のくさっ...


In [10]:
# 重複データを削除
df = df.drop_duplicates(subset=['title', 'body']).reset_index(drop=True)

# 重複を削除した後のデータ数を確認
print("重複削除後のデータ数:", len(df))

重複削除後のデータ数: 7370


# **データ分割**

In [9]:
# fugashiとipadicをインストール
!pip install fugashi ipadic
!pip install unidic-lite

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 80.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 698.4/698.4 kB 33.9 MB/s eta 0:00:00
  Created wheel for ipadic: filename=ipadic-1.0.0-py3-none-any.whl size=13556704 sha256=614126be194907dc61fc594e4c062ff8bfdd4ba2fea3fd3211aeed9b392c304c
  Stored in directory: /root/.cache/pip/wheels/44/56/37/f543963822b85260c9f948df8fac8c20169c80dc71b24dc407
Successfully built ipadic
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658817 sha256=7ec2bbfa0245ce8566c83123ea3142dc3283f6cd49c74829a1756caddcfbed8c
  Stored in directory: /root/.cache/pip/wheels/b7/fd/e9/ea4459b868e6d2902e8d80e82dbacb6203e05b3b3a58c64966
Successfully built unidic-lite


試験用
訓練データと検証データ、テストデータに分けておく

12/2 これは本番で使う物

2024 12/7 カテゴリごとにデータを選んでTrain, Validation, Testの3つのデータフレームを作成
具体例（仮定: dokujo-tsushinのデータが700件の場合）
元データ: 700件訓練データ → 30件
残り → 670件検証データ → 30件
残り → 640件テストデータ → 30件

In [ ]:
import pandas as pd

# カテゴリごとに均等にデータを分割する関数
def split_data_by_category(df, samples_per_category=30):
    train_set = pd.DataFrame()
    validation_set = pd.DataFrame()
    test_set = pd.DataFrame()

    # ユニークなカテゴリごとに分割
    categories = df['category'].unique()

    for category in categories:
        # カテゴリごとのデータを取得
        category_data = df[df['category'] == category]

        # データが十分な件数を持っているか確認
        if len(category_data) < 3 * samples_per_category:
            raise ValueError(f"カテゴリ '{category}' に十分なデータがありません。")

        # 訓練、検証、テストデータをランダムに分割
        train_data = category_data.sample(n=samples_per_category, random_state=42)
        remaining_data = category_data.drop(train_data.index)

        validation_data = remaining_data.sample(n=samples_per_category, random_state=42)

        test_data = remaining_data.drop(validation_data.index).sample(n=samples_per_category, random_state=42)

        # 各セットに追加
        train_set = pd.concat([train_set, train_data], axis=0)
        validation_set = pd.concat([validation_set, validation_data], axis=0)
        test_set = pd.concat([test_set, test_data], axis=0)

    return train_set, validation_set, test_set


# データフレーム df を元に分割を実行
train_df, validation_df, test_df = split_data_by_category(df)

# 結果の確認
print("=== データセット概要 ===")
print("訓練データの件数:", len(train_df))
print("検証データの件数:", len(validation_df))
print("テストデータの件数:", len(test_df))

print("\n=== カラム情報 ===")
print("訓練データのカラム:\n", train_df.columns.tolist())

print("\n=== カテゴリ分布 ===")
print("訓練データのカテゴリ分布:\n", train_df['category'].value_counts())
print("検証データのカテゴリ分布:\n", validation_df['category'].value_counts())
print("テストデータのカテゴリ分布:\n", test_df['category'].value_counts())

print("\n=== 訓練データサンプル ===")
print(train_df.head())

=== データセット概要 ===
訓練データの件数: 270
検証データの件数: 270
テストデータの件数: 270

=== カラム情報 ===
訓練データのカラム:
 ['category', 'title', 'body']

=== カテゴリ分布 ===
訓練データのカテゴリ分布:
 category
livedoor-homme    30
peachy            30
movie-enter       30
topic-news        30
smax              30
dokujo-tsushin    30
it-life-hack      30
sports-watch      30
kaden-channel     30
Name: count, dtype: int64
検証データのカテゴリ分布:
 category
livedoor-homme    30
peachy            30
movie-enter       30
topic-news        30
smax              30
dokujo-tsushin    30
it-life-hack      30
sports-watch      30
kaden-channel     30
Name: count, dtype: int64
テストデータのカテゴリ分布:
 category
livedoor-homme    30
peachy            30
movie-enter       30
topic-news        30
smax              30
dokujo-tsushin    30
it-life-hack      30
sports-watch      30
kaden-channel     30
Name: count, dtype: int64

=== 訓練データサンプル ===
           category                                    title  \
304  livedoor-homme              伝統のバーボンウイスキーに学ぶコミュニケーションの秘訣   
49

In [ ]:
print(train_df["body"].apply(len).describe())
print(train_df[train_df["body"].str.strip() == ""])

count     270.000000
mean     1106.559259
std       651.419835
min        38.000000
25%       620.250000
50%       944.500000
75%      1405.000000
max      4588.000000
Name: body, dtype: float64
Empty DataFrame
Columns: [category, title, body]
Index: []


# データ拡張

Bertによる拡張

In [ ]:
!pip install fugashi
!pip install ipadic
!apt-get install -y mecab mecab-ipadic-utf8 libmecab-dev swig
!pip install mecab-python3 fugashi ipadic
!pip install unidic-lite

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libmecab2 mecab-ipadic mecab-utils swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  libmecab-dev libmecab2 mecab mecab-ipadic mecab-ipadic-utf8 mecab-utils swig swig4.0
0 upgraded, 8 newly installed, 0 to remove and 49 not upgraded.
Need to get 8,483 kB of archives.
After this operation, 64.8 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libmecab2 amd64 0.996-14build9 [199 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libmecab-dev amd64 0.996-14build9 [306 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 mecab-utils amd64 0.996-14build9 [4,850 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 mecab-ipadic all 2.7.0-20070801+main-3 [6,718 kB]
Get:5 http://archive.ubuntu.com

1. 必要なライブラリとモデルの準備
まずは、必要なライブラリをインポートして、BERTモデルやトークナイザを準備する。

bodyテキストを拡張する関数を作成します。

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import random

# GPUまたはCPUの設定
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_names = {
    "tohoku_bert": "cl-tohoku/bert-base-japanese",
    "tohoku_bert_v2": "cl-tohoku/bert-base-japanese-v2",
    "stockmark_bert": "stockmark/bart-base-japanese-news",
}

# トークナイザとモデルのロード
models = {}
tokenizers = {}

for name, model_path in model_names.items():
    tokenizers[name] = AutoTokenizer.from_pretrained(model_path)
    models[name] = AutoModelForMaskedLM.from_pretrained(model_path).to(device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/258k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/stockmark/bart-base-japanese-news:
- tokenization_bart_japanese_news.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


spiece.model:   0%|          | 0.00/828k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [ ]:
def augment_text(text, tokenizer, model, max_length=512):
    # テキストをトークン化
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(device)
    input_ids = inputs["input_ids"][0]

    # 短いテキストはそのまま返す
    if len(input_ids) < 3:
        return text

    # ランダムな位置をマスク
    mask_pos = random.randint(1, len(input_ids) - 2)  # [CLS]や[SEP]を避ける
    input_ids[mask_pos] = tokenizer.mask_token_id

    # マスクされた入力をモデルに通す
    with torch.no_grad():
        outputs = model(input_ids=input_ids.unsqueeze(0))
    predictions = outputs.logits

    # マスクされた位置の予測トークンを取得
    predicted_token_id = predictions[0, mask_pos].argmax(dim=-1).item()

    # 予測トークンで置換して再構築
    input_ids[mask_pos] = predicted_token_id
    augmented_text = tokenizer.decode(input_ids, skip_special_tokens=True)
    return augmented_text

データフレームを拡張

In [ ]:
# データ拡張ループ
for model_name in model_names.keys():
    print(f"拡張中: {model_name}")
    tokenizer = tokenizers[model_name]
    model = models[model_name]

    def debug_augment(text):
        if not text or len(text.strip()) < 2:
            print("短すぎる入力:", text)
            return text
        try:
            if model_name == "rinna_roberta":
                return augment_with_rinna(text, tokenizer, model)  # rinna専用の処理
            else:
                return augment_text(text, tokenizer, model)  # 通常処理
        except Exception as e:
            print(f"エラー発生: {model_name} - 入力: {text[:30]}")
            print(f"例外内容: {e}")
            return text  # エラー時はオリジナルを返す

    # データ拡張
    train_df[f"augmented_body_{model_name}"] = train_df["body"].apply(debug_augment)

拡張中: tohoku_bert
拡張中: tohoku_bert_v2
拡張中: stockmark_bert


In [ ]:
# トークナイザを用いてトークン数を確認
for model_name, tokenizer in tokenizers.items():
    print(f"トークン数確認中: {model_name}")
    train_df[f"token_count_{model_name}"] = train_df["body"].apply(
        lambda x: len(tokenizer.tokenize(x))
    )
    print(train_df[f"token_count_{model_name}"].describe())

トークン数確認中: tohoku_bert
count     270.000000
mean      628.974074
std       369.330696
min        14.000000
25%       354.250000
50%       513.000000
75%       830.750000
max      2683.000000
Name: token_count_tohoku_bert, dtype: float64
トークン数確認中: tohoku_bert_v2
count     270.000000
mean      628.488889
std       368.829816
min        13.000000
25%       356.000000
50%       510.000000
75%       830.250000
max      2706.000000
Name: token_count_tohoku_bert_v2, dtype: float64
トークン数確認中: stockmark_bert
count     270.000000
mean      463.800000
std       265.495099
min        14.000000
25%       258.250000
50%       394.500000
75%       595.000000
max      2014.000000
Name: token_count_stockmark_bert, dtype: float64


結果

In [ ]:
print("拡張後のデータフレーム:")
print(train_df.head())

print("\n新しいカラム情報:")
print(train_df.columns.tolist())

拡張後のデータフレーム:
           category                                    title  \
304  livedoor-homme              伝統のバーボンウイスキーに学ぶコミュニケーションの秘訣   
495  livedoor-homme   動画あり社長がドリフト  世界のTOYOTAが本気で作った86がいよいよ発売   
439  livedoor-homme      新生活にインテリアを一新イケア流トータルコーディネート術とは新生活特集   
153  livedoor-homme              イマドキOLのホンネ座談会スマートなオトコの新基準とは   
497  livedoor-homme  ポテンシャルが高い人はココが違う人事担当者がこっそり教える採用ウラ話 vol2   

                                                  body  \
304  時代の移り変わりとともにコミュニケーションの手段は変わっていくものそして近年のコミュニケーシ...   
495  カッコいい\nそんなセリフがおもわず口をついて出てしまうほど鮮烈な印象を与えたのが2009年...   
439  スウェーデン生まれのホームファニッシングストアとして爆発的な人気を誇るイケア今年で日本上陸4...   
153  先日発表になった2011年の冬のボーナスの見通しによれば民間企業の一人当たりの平均支給額は3...   
497  転職者なら誰でも気になる採用する側の心理しかし実際に人事担当者から本音を聞きだすことはなかな...   

                            augmented_body_tohoku_bert  \
304  時代 の 移り変わり と とも に コミュニケーション の 手段 は 変わっ て いく もの...   
495  カッコ いい そんな セリフ が おもわ ず 口 を つい て 出 て しまう ほど 鮮烈 ...   
439  スウェーデン 生まれ の ホームファニッシングストア と し て 爆発 的 な 人気 を 誇...   
153  先日 発表 に なっ た 201

In [ ]:
# 元データにフラグ追加
df_original = train_df.copy()
df_original["is_augmented"] = "original"

# 東北BERT拡張データを作成
df_tohoku = train_df.copy()
df_tohoku["body"] = df_tohoku["augmented_body_tohoku_bert"]
df_tohoku["is_augmented"] = "tohoku_bert"

# 元データ + 拡張データを結合
df_tohoku_combined = pd.concat([df_original, df_tohoku], axis=0).reset_index(drop=True)

In [ ]:
# 東北BERTv2拡張データを作成
df_tohoku_v2 = train_df.copy()
df_tohoku_v2["body"] = df_tohoku_v2["augmented_body_tohoku_bert_v2"]
df_tohoku_v2["is_augmented"] = "tohoku_bert_v2"

# 元データ + 拡張データを結合
df_tohoku_v2_combined = pd.concat([df_original, df_tohoku_v2], axis=0).reset_index(drop=True)

In [ ]:
# Stockmark BERT拡張データを作成
df_stockmark = train_df.copy()
df_stockmark["body"] = df_stockmark["augmented_body_stockmark_bert"]
df_stockmark["is_augmented"] = "stockmark_bert"

# 元データ + 拡張データを結合
df_stockmark_combined = pd.concat([df_original, df_stockmark], axis=0).reset_index(drop=True)

In [ ]:
# 削除するカラムのリスト
columns_to_drop = [
    'augmented_body_tohoku_bert',
    'augmented_body_tohoku_bert_v2',
    'augmented_body_stockmark_bert',
    'token_count_tohoku_bert',
    'token_count_tohoku_bert_v2',
    'token_count_stockmark_bert'
]

# 各データフレームから不要カラムを削除
df_original = df_original.drop(columns=columns_to_drop, errors='ignore')
df_tohoku_combined = df_tohoku_combined.drop(columns=columns_to_drop, errors='ignore')
df_tohoku_v2_combined = df_tohoku_v2_combined.drop(columns=columns_to_drop, errors='ignore')
df_stockmark_combined = df_stockmark_combined.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
print(f"元のデータ数: {len(df_original)}")
print(f"東北BERT（結合後）: {len(df_tohoku_combined)}")
print(f"東北BERTv2（結合後）: {len(df_tohoku_v2_combined)}")
print(f"Stockmark BERT（結合後）: {len(df_stockmark_combined)}")

# 各データフレームの一部を確認
print(df_original.columns.tolist())
print(df_tohoku_combined.columns.tolist())
print(df_tohoku_v2_combined.columns.tolist())
print(df_stockmark_combined.columns.tolist())

# サンプルデータを表示
print(df_tohoku_v2_combined.head())
print(df_tohoku_v2_combined.tail())  # 結合された拡張データを確認

元のデータ数: 270
東北BERT（結合後）: 540
東北BERTv2（結合後）: 540
Stockmark BERT（結合後）: 540
['category', 'title', 'body', 'is_augmented', 'label']
['category', 'title', 'body', 'is_augmented', 'label']
['category', 'title', 'body', 'is_augmented', 'label']
['category', 'title', 'body', 'is_augmented', 'label']
         category                                              title  \
0          peachy                             イチゴ1月15日の日に限定コラボスイーツ誕生   
1    it-life-hack                最新のFirefoxが登場 Firefox 11日本語版が公式リリース   
2   kaden-channel          自殺をほのめかすユーザーは即通報 Facebookが自殺防止の取り組みスタート話題   
3  livedoor-homme  フルレンジサウンドの魅力を追求し様々なテクノロジーを投入したプレミアムな逸品ウッドコーン特別...   
4  livedoor-homme                                 新生活には目覚めのよい朝を新生活特集   

                                                body    is_augmented  label  
0  14 日 東京 自由が丘 スイーツ フォレスト で 静岡 県 で NO 1 シェア を 誇る...  tohoku_bert_v2      1  
1  Mozzila Japanは新しいWebブラウザとなるFirefox 11をリリースした前バ...        original      6  
2  自殺 者 の 増加 が 問題 に なっ て いる 日本 で も 自殺 者 

KeyError: 'list'

拡張データを元データに結合してデータ量を増やす：

In [ ]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ファイルの保存先パス
file_path = '/content/drive/My Drive/train_df.csv'

# DataFrameを保存
train_df.to_csv(file_path, index=False)

# 復元する場合
import pandas as pd
train_df = pd.read_csv(file_path)

MessageError: Error: credential propagation was unsuccessful

# **データ分類**


 ▼前提として、train_df, validation_df, test_dfがPandas DataFrameで読み込まれているとする
train_df: category, title, body, augmented_body_tohoku_bert, ... validation_df: category, title, body
test_df: category, title, body
categoryはラベル、augmented_body_tohoku_bertは拡張テキスト
validation_df, test_dfは拡張なし
ここでは category を整数ラベルに変換して使う想定
categoryが既に文字列クラスなら、ユニーク値からidにマップする必要がある

In [ ]:
# カテゴリのユニーク値からラベルを作成
label_list = df_original['category'].unique().tolist()
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

In [ ]:
# 元のデータフレームにラベルを追加
df_original['label'] = df_original['category'].map(label2id)

# 統合後のデータフレームにもラベルを追加
df_tohoku_combined['label'] = df_tohoku_combined['category'].map(label2id)
df_tohoku_v2_combined['label'] = df_tohoku_v2_combined['category'].map(label2id)
df_stockmark_combined['label'] = df_stockmark_combined['category'].map(label2id)

validation_df['label'] = validation_df['category'].map(label2id)
test_df['label'] = test_df['category'].map(label2id)

In [ ]:
print(label2id)  # {'ニュース': 0, 'スポーツ': 1, '経済': 2} のような形式
print(id2label)  # {0: 'ニュース', 1: 'スポーツ', 2: '経済'} のような形式

{'livedoor-homme': 0, 'peachy': 1, 'movie-enter': 2, 'topic-news': 3, 'smax': 4, 'dokujo-tsushin': 5, 'it-life-hack': 6, 'sports-watch': 7, 'kaden-channel': 8}
{0: 'livedoor-homme', 1: 'peachy', 2: 'movie-enter', 3: 'topic-news', 4: 'smax', 5: 'dokujo-tsushin', 6: 'it-life-hack', 7: 'sports-watch', 8: 'kaden-channel'}


In [ ]:
print(df_original.head())
print(df_tohoku_combined.head())
print(df_tohoku_v2_combined.head())
print(df_stockmark_combined.head())
print(validation_df.head())
print(test_df.head())

print("\n=== リスト分布 ===")
print("訓練データのカテゴリ分布:\n", df_original['list'].value_counts())
print("検証データのカテゴリ分布:\n", validation_df['list'].value_counts())
print("テストデータのカテゴリ分布:\n", test_df['list'].value_counts())

           category                                    title  \
304  livedoor-homme              伝統のバーボンウイスキーに学ぶコミュニケーションの秘訣   
495  livedoor-homme   動画あり社長がドリフト  世界のTOYOTAが本気で作った86がいよいよ発売   
439  livedoor-homme      新生活にインテリアを一新イケア流トータルコーディネート術とは新生活特集   
153  livedoor-homme              イマドキOLのホンネ座談会スマートなオトコの新基準とは   
497  livedoor-homme  ポテンシャルが高い人はココが違う人事担当者がこっそり教える採用ウラ話 vol2   

                                                  body is_augmented  label  
304  時代の移り変わりとともにコミュニケーションの手段は変わっていくものそして近年のコミュニケーシ...     original      0  
495  カッコいい\nそんなセリフがおもわず口をついて出てしまうほど鮮烈な印象を与えたのが2009年...     original      0  
439  スウェーデン生まれのホームファニッシングストアとして爆発的な人気を誇るイケア今年で日本上陸4...     original      0  
153  先日発表になった2011年の冬のボーナスの見通しによれば民間企業の一人当たりの平均支給額は3...     original      0  
497  転職者なら誰でも気になる採用する側の心理しかし実際に人事担当者から本音を聞きだすことはなかな...     original      0  
         category                                    title  \
0  livedoor-homme              伝統のバーボンウイスキーに学ぶコミュニケーションの秘訣   
1  livedoor-homme   動画あり社長がドリ

In [ ]:
print("Train DataFrame Columns:", df_original.columns)
print("Validation DataFrame Columns:", validation_df.columns)
print("Test DataFrame Columns:", test_df.columns)

Train DataFrame Columns: Index(['category', 'title', 'body', 'is_augmented', 'label'], dtype='object')
Validation DataFrame Columns: Index(['category', 'title', 'body', 'label'], dtype='object')
Test DataFrame Columns: Index(['category', 'title', 'body', 'label'], dtype='object')


In [ ]:
# サンプルデータを確認
print(df_original[['category', 'label']].head())
print(df_tohoku_combined[['category', 'label']].tail())

           category  label
304  livedoor-homme      0
495  livedoor-homme      0
439  livedoor-homme      0
153  livedoor-homme      0
497  livedoor-homme      0
          category  label
535  kaden-channel      8
536  kaden-channel      8
537  kaden-channel      8
538  kaden-channel      8
539  kaden-channel      8


In [ ]:
!pip install transformers datasets sentencepiece evaluate
!pip install evaluate

import pandas as pd
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments
from evaluate import load
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 17.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
# メトリクス計算用関数
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    print(logits[0])
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    # print(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, df, text_cols, tokenizer, max_length=128):
        self.df = df.reset_index(drop=True)
        self.text_cols = text_cols
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # 2つのテキスト列(category, body)を想定
        text = ""
        for col in self.text_cols:
            text += str(row[col]) + " "

        label = row["label"]
        encoding = self.tokenizer(
            text.strip(),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": label
        }

In [ ]:
# 事前学習モデル名
model_name = "cl-tohoku/bert-base-japanese-whole-word-masking"
tokenizer = BertTokenizer.from_pretrained(model_name)

# この時点で df_original, validation_df, test_df, df_tohoku_combined, df_tohoku_v2_combined, df_stockmark_combined が存在し
# かつ "category", "body", "label" を含むとする

# num_labels計算
num_labels = df_original["label"].nunique()

# モデル作成
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

tokenizer_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/258k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertJapaneseTokenizer'. 
The class this function is called from is 'BertTokenizer'.


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cl-tohoku/bert-base-japanese-whole-word-masking and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def train_and_evaluate(train_df, run_name):
    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

    # データセット作成
    train_dataset = NewsDataset(train_df, ["category", "body"], tokenizer)
    val_dataset = NewsDataset(validation_df, ["category", "body"], tokenizer)
    test_dataset = NewsDataset(test_df, ["category", "body"], tokenizer)

    training_args = TrainingArguments(
        output_dir=f"./results_{run_name}",
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        evaluation_strategy="steps",
        save_steps=100,
        eval_steps=100,
        logging_steps=50,
        learning_rate=1e-3,
        num_train_epochs=5,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    trainer.train()

    # バリデーションデータで評価
    val_metrics = trainer.evaluate(val_dataset)
    print(f"=== {run_name} Validation ===")
    print(val_metrics)

    # テストデータで評価
    test_metrics = trainer.evaluate(test_dataset)
    print(f"=== {run_name} Test ===")
    print(test_metrics)
    return val_metrics, test_metrics

In [ ]:
# データフレームをシャッフルする（例: random_state=42は再現性のための固定シード値）
df_original = df_original.sample(frac=1.0, random_state=42).reset_index(drop=True)
df_tohoku_combined = df_tohoku_combined.sample(frac=1.0, random_state=42).reset_index(drop=True)
df_tohoku_v2_combined = df_tohoku_v2_combined.sample(frac=1.0, random_state=42).reset_index(drop=True)
df_stockmark_combined = df_stockmark_combined.sample(frac=1.0, random_state=42).reset_index(drop=True)

# その後に4パターンの学習を実行
results_original = train_and_evaluate(df_original, "Original Dataset")
results_tohoku = train_and_evaluate(df_tohoku_combined, "Tohoku Combined Dataset")
results_tohoku_v2 = train_and_evaluate(df_tohoku_v2_combined, "Tohoku v2 Combined Dataset")
results_stockmark = train_and_evaluate(df_stockmark_combined, "Stockmark Combined Dataset")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cl-tohoku/bert-base-japanese-whole-word-masking and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-43-4ec6e82601d0>:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
100,2.510900,2.476694,0.111111,0.012346,0.111111,0.022222
200,2.540100,2.436880,0.111111,0.012346,0.111111,0.022222
300,2.521000,2.414148,0.111111,0.012346,0.111111,0.022222
400,2.416600,2.264763,0.111111,0.012346,0.111111,0.022222
500,2.320500,2.372527,0.111111,0.012346,0.111111,0.022222
600,2.282000,2.228726,0.111111,0.012346,0.111111,0.022222


[-0.78231823 -0.38103637  0.1079023   0.92933834 -1.6661752  -0.19832647
 -0.39669845 -1.487024   -0.5829706 ]
[-0.8221866  -1.4998047   0.23505048 -1.1372908  -1.075213    0.74153256
  0.08831696 -0.34621048 -0.40728435]
[-0.66053295  1.0102623  -0.648241   -0.6157333  -0.80840427 -0.23058634
 -0.4257966  -0.9908158  -0.78465044]
[-0.71134365 -0.9372739  -0.3868441  -0.71070915  0.18548714 -0.8426203
 -0.52103424 -0.00492229 -0.28482348]
[ 0.26327753  0.13353445 -1.4706608  -1.2124196  -0.6187256  -0.33250248
  0.23805314 -0.08938237 -1.076591  ]
[-0.17264116 -0.1481787  -0.8429505  -0.17703453 -0.6758053  -0.6764865
 -0.3775016  -0.71513754 -0.43451998]


[-0.78231823 -0.38103637  0.1079023   0.92933834 -1.6661752  -0.19832647
 -0.39669845 -1.487024   -0.5829706 ]
=== Original Dataset Validation ===
{'eval_loss': 2.476693868637085, 'eval_accuracy': 0.1111111111111111, 'eval_precision': 0.012345679012345678, 'eval_recall': 0.1111111111111111, 'eval_f1': 0.022222222222222223, 'eval_runtime': 3.4171, 'eval_samples_per_second': 79.015, 'eval_steps_per_second': 39.508, 'epoch': 5.0}
[-0.78231835 -0.38103643  0.10790204  0.9293387  -1.6661752  -0.19832645
 -0.39669845 -1.487024   -0.58297056]
=== Original Dataset Test ===
{'eval_loss': 2.476693868637085, 'eval_accuracy': 0.1111111111111111, 'eval_precision': 0.012345679012345678, 'eval_recall': 0.1111111111111111, 'eval_f1': 0.022222222222222223, 'eval_runtime': 3.6071, 'eval_samples_per_second': 74.852, 'eval_steps_per_second': 37.426, 'epoch': 5.0}


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cl-tohoku/bert-base-japanese-whole-word-masking and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-43-4ec6e82601d0>:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

[-1.1038347  -0.73756224  0.6474498  -0.00471318 -2.213227   -2.7142942
 -0.5328933  -3.83062    -0.37907687]
=== Tohoku v2 Combined Dataset Validation ===
{'eval_loss': 2.8533198833465576, 'eval_accuracy': 0.1111111111111111, 'eval_precision': 0.012345679012345678, 'eval_recall': 0.1111111111111111, 'eval_f1': 0.022222222222222223, 'eval_runtime': 4.5679, 'eval_samples_per_second': 59.108, 'eval_steps_per_second': 29.554, 'epoch': 5.0}
[-1.1038347  -0.73756224  0.6474498  -0.00471318 -2.213227   -2.7142942
 -0.5328933  -3.83062    -0.37907687]
=== Tohoku v2 Combined Dataset Test ===
{'eval_loss': 2.8533198833465576, 'eval_accuracy': 0.1111111111111111, 'eval_precision': 0.012345679012345678, 'eval_recall': 0.1111111111111111, 'eval_f1': 0.022222222222222223, 'eval_runtime': 3.6279, 'eval_samples_per_second': 74.424, 'eval_steps_per_second': 37.212, 'epoch': 5.0}


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cl-tohoku/bert-base-japanese-whole-word-masking and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-43-4ec6e82601d0>:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
100,2.644400,2.915381,0.111111,0.012346,0.111111,0.022222
200,2.538900,2.512726,0.111111,0.012346,0.111111,0.022222
300,2.590800,2.910777,0.111111,0.012346,0.111111,0.022222
400,2.400800,2.439559,0.111111,0.012346,0.111111,0.022222
500,2.600700,2.265785,0.111111,0.012346,0.111111,0.022222
600,2.442300,2.306051,0.111111,0.012346,0.111111,0.022222
700,2.477600,2.420429,0.111111,0.012346,0.111111,0.022222
800,2.394900,2.285067,0.111111,0.012346,0.111111,0.022222
900,2.374000,2.259779,0.111111,0.012346,0.111111,0.022222
1000,2.312800,2.257325,0.111111,0.012346,0.111111,0.022222


[ 1.368646    1.5197325   3.1142945   2.3342133   0.02760901 -0.29258293
  2.0999014  -1.7416912   1.9481406 ]
[ 1.9149593   1.6502806   0.6137821  -0.09289841  0.4986143   2.5970519
  1.3467758   0.634313    1.8676748 ]
[-0.27106982  3.1121304  -0.21630934  1.2953408   2.1933916   2.8920975
  1.0571231   0.97746104 -0.23782964]
[0.419175   1.1925126  1.4233562  1.3881325  0.33010942 1.0013244
 1.1175866  1.4056659  2.67419   ]
[1.2908174  0.36293346 1.1972369  1.0618535  1.4698642  1.0434989
 1.3174407  1.8650361  1.4771587 ]
[1.0198474  1.6692079  1.4281901  1.1990076  0.9574206  0.9732
 1.4621985  2.0703654  0.31869197]
[0.5310439  1.707359   1.8766856  0.42803824 0.36404753 1.1986754
 0.9084001  1.816512   2.319265  ]
[1.4155419  1.3563672  0.76608604 0.6211235  0.5826786  1.5075934
 1.4194429  1.6500978  1.8373604 ]
[0.9077947  1.6548973  0.86398137 0.76121145 1.1395111  1.4008842
 1.7913234  1.0868168  1.5617259 ]
[0.92275685 1.1831205  1.065208   1.4231063  1.771661   0.46798497

[ 1.368646    1.5197325   3.1142945   2.3342133   0.02760901 -0.29258293
  2.0999014  -1.7416912   1.9481406 ]
=== Stockmark Combined Dataset Validation ===
{'eval_loss': 2.91538143157959, 'eval_accuracy': 0.1111111111111111, 'eval_precision': 0.012345679012345678, 'eval_recall': 0.1111111111111111, 'eval_f1': 0.022222222222222223, 'eval_runtime': 3.7383, 'eval_samples_per_second': 72.225, 'eval_steps_per_second': 36.113, 'epoch': 5.0}
[ 1.368646    1.5197325   3.1142945   2.3342133   0.02760901 -0.29258293
  2.0999014  -1.7416912   1.9481406 ]
=== Stockmark Combined Dataset Test ===
{'eval_loss': 2.91538143157959, 'eval_accuracy': 0.1111111111111111, 'eval_precision': 0.012345679012345678, 'eval_recall': 0.1111111111111111, 'eval_f1': 0.022222222222222223, 'eval_runtime': 4.4693, 'eval_samples_per_second': 60.412, 'eval_steps_per_second': 30.206, 'epoch': 5.0}
